# 重排特征与训练选择

自主训练代码位于 src/ragstudy/experiment.py。每个训练折独立拟合标准化；全部候选随 query 所属组件进出验证折。未标注候选只是代理负例。

In [1]:
from pathlib import Path
import json, sys, os
import pandas as pd
from IPython.display import display, Image
root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'project/src/ragstudy').is_dir())
for folder in ('.vendor', '.vendor-hf', '.vendor-onnx-compat'):
    path = root / 'project' / folder
    if path.is_dir():
        sys.path.insert(0, str(path))
sys.path.insert(0, str(root / 'project/src'))
artifacts = sorted(path for path in (root / 'project/artifacts').iterdir()
                   if (path / 'results.json').is_file() and (path / 'search_bundle.joblib').is_file())
assert artifacts, 'First run python project/scripts/run_experiment.py --backend lsa'
artifact = Path(os.environ['RAG_ARTIFACT']).resolve() if os.environ.get('RAG_ARTIFACT') else artifacts[-1]
result = json.loads((artifact / 'results.json').read_text(encoding='utf-8'))
print('Actual local run:', artifact.name, '| backend:', result['retrieval_metadata']['semantic']['method'])

Actual local run: frozen_e2324b7a967c5dbe2766ecc3 | backend: frozen_onnx_fp32_cls_cosine


In [2]:
display(pd.DataFrame(result['cross_validation']))
display(pd.DataFrame(result['selected_models']).T[['parameters','training_pairs','cv_ndcg@10']])
print('Features:', result['feature_names'])

,model,parameters,cv_ndcg@10,queries,folds
0,logistic,{'C': 0.1},0.601376,410,3
1,logistic,{'C': 1.0},0.599705,410,3
2,logistic,{'C': 10.0},0.599035,410,3
3,random_forest,"{'max_depth': 6, 'min_samples_leaf': 10}",0.590955,410,3
4,random_forest,"{'max_depth': 6, 'min_samples_leaf': 30}",0.589491,410,3
5,random_forest,"{'max_depth': 12, 'min_samples_leaf': 10}",0.591090,410,3
6,random_forest,"{'max_depth': 12, 'min_samples_leaf': 30}",0.588515,410,3


,parameters,training_pairs,cv_ndcg@10
logistic,{'C': 0.1},18768,0.601376
random_forest,"{'max_depth': 12, 'min_samples_leaf': 10}",18768,0.59109


Features: ['bm25_score', 'semantic_score', 'bm25_reciprocal_rank', 'semantic_reciprocal_rank', 'missing_bm25_rank', 'missing_semantic_rank', 'query_term_coverage', 'log_query_length', 'log_document_length']


In [3]:
from ragstudy.search import load_bundle
bundle = load_bundle(artifact)
model_bundle = bundle['models']['logistic']
model = model_bundle['model']
selected_features = [result['feature_names'][i] for i in model_bundle.get('feature_indices', range(len(result['feature_names'])))]
display(pd.DataFrame({'feature':selected_features, 'standardized_logistic_coefficient':model.coef_[0]}))
print('Scores support ranking; they are not calibrated relevance probabilities.')

,feature,standardized_logistic_coefficient
0,bm25_score,0.104265
1,semantic_score,0.017010
2,bm25_reciprocal_rank,0.430003
3,semantic_reciprocal_rank,0.472018
4,missing_bm25_rank,-0.398526
5,missing_semantic_rank,-0.854364
6,query_term_coverage,-0.038478
7,log_query_length,-0.013212
8,log_document_length,0.008501


Scores support ranking; they are not calibrated relevance probabilities.
